In [ ]:
import os

import cmcrameri.cm as cmc
import numpy as np
import torch
import yaml
from matplotlib import colors
from matplotlib import pyplot as plt
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar
from mpl_toolkits.axes_grid1.axes_divider import make_axes_locatable
from ptychi.image_proc import place_patches_fourier_shift

from ptycho_fm.data import PtychographyDataset

%matplotlib inline

## Already stitched inference

In [ ]:
object_name = 'scan_285'

# Load results
results_dir = '/mnt/localdisk/scratch/aileenluo/results/run316'

pred_amp_test = np.load(os.path.join(results_dir, f'pred_amp_object_{object_name}_crop96.npy'))
pred_ph_test = np.load(os.path.join(results_dir, f'pred_ph_object_{object_name}_crop96.npy'))

pred_amp_test.shape

In [ ]:
f, ax = plt.subplots(figsize=(8, 3), ncols=2, dpi=300)

amp = ax[0].imshow(pred_amp_test, interpolation='none', cmap=cmc.oslo, vmin=0.8, origin='lower')
divider = make_axes_locatable(ax[0])
cax = divider.append_axes('right', size='5%', pad=0.05)
f.colorbar(amp, cax=cax, orientation='vertical')
ax[0].set_xticks([])
ax[0].set_yticks([])
ax[0].set_title('Amplitude')

phi = ax[1].imshow(pred_ph_test, interpolation='none', cmap='Grays_r', origin='lower', vmin=-0.9, vmax=-0.5)
divider = make_axes_locatable(ax[1])
cax = divider.append_axes('right', size='5%', pad=0.05)
f.colorbar(phi, cax=cax, orientation='vertical')
scalebar = AnchoredSizeBar(ax[1].transData, 299, '$2 \mu m$', 'lower right', pad=0.1, color='black', frameon=False, size_vertical=5)
ax[1].add_artist(scalebar)
ax[1].set_xticks([])
ax[1].set_yticks([])
ax[1].set_title('Phase')

For incomplete inference (i.e. without predicted diffraction amplitude), do not uncomment pred_diff_test

In [ ]:
object_name = 'scan807'
run_num = 130
test_data_dir = "/scratch/aileenluo/ptycho-vit/test_exp"
results_dir = '/scratch/aileenluo/ptycho-vit/results/run{run_num}'

pred_amp_test = np.load(os.path.join(results_dir, f'pred_amp_{object_name}.npy'))
pred_ph_test = np.load(os.path.join(results_dir, f'pred_ph_{object_name}.npy'))

# Optional: load predicted diffraction amplitudes as well (this is slow!)
# pred_diff_test = np.load(os.path.join(results_dir, f'pred_diff_{object_name}.npy'))

In [ ]:
config_path = f'/scratch/aileenluo/ptycho-vit/models/run{run_num}/config.yaml'

def load_config(path):
    """Load configuration from YAML file."""
    with open(path, 'r') as f:
        config = yaml.safe_load(f)
        return config

config = load_config(config_path)

In [ ]:
# Data for object patch placement using object_name
from pathlib import Path

# Find normalization pkl file in test data directory
pkl_files = list(Path(test_data_dir).glob('*.pkl'))
normalization_dict_path = str(pkl_files[0]) if pkl_files else None

dataset = PtychographyDataset(
    os.path.join(test_data_dir, f'{object_name}_dp.hdf5'), 
    config['data']['scale'],
    normalization_dict_path,
    apply_noise=False, # Use True for simulated data
    cache_object=config['data'].get('cache_object', True),
    cache_memory_budget_mb=config['data'].get('cache_memory_budget_mb', 512),
    max_probe_modes=config['data'].get('max_probe_modes', 8),
    max_OPR_modes=config['data'].get('max_OPR_modes', 1),
)

gt_diff_test = torch.zeros((len(dataset), dataset.pattern_shape[0], dataset.pattern_shape[1]))
gt_amp_test = torch.zeros((len(dataset), dataset.pattern_shape[0], dataset.pattern_shape[1]))
gt_ph_test = torch.zeros((len(dataset), dataset.pattern_shape[0], dataset.pattern_shape[1]))
for i in range(len(dataset)):
    gt_diff_test[i], gt_amp_test[i], gt_ph_test[i], _probe, _position, _norm, _scale = dataset[i]

positions = dataset.get_probe_positions()
object_size = dataset.object_shape
object_size

In [ ]:
central_crop = 64
pred_amp_object = torch.zeros(object_size)
pred_ph_object = torch.zeros(object_size)
buffer = torch.zeros(object_size)

pred_amp_object = place_patches_fourier_shift(
    pred_amp_object, 
    positions, 
    torch.from_numpy(pred_amp_test[:, central_crop:-central_crop, central_crop:-central_crop]), 
    "add", 
    False, 
    pad=32
    )
pred_ph_object = place_patches_fourier_shift(
    pred_ph_object, 
    positions, 
    torch.from_numpy(pred_ph_test[:, central_crop:-central_crop, central_crop:-central_crop]), 
    "add", 
    False, 
    pad=32
    )
buffer = place_patches_fourier_shift(
    buffer, 
    positions, 
    torch.ones_like(torch.from_numpy(pred_ph_test[:, central_crop:-central_crop, central_crop:-central_crop])), 
    "add", 
    False, 
    pad=32
    )

gt_amp_object = torch.zeros(object_size)
gt_ph_object = torch.zeros(object_size)

gt_amp_object = place_patches_fourier_shift(
    gt_amp_object, 
    positions, 
    torch.from_numpy(gt_amp_test[:, central_crop:-central_crop, central_crop:-central_crop]), 
    "add", 
    False,
    pad=32
    )
gt_ph_object = place_patches_fourier_shift(
    gt_ph_object, 
    positions, 
    torch.from_numpy(gt_ph_test[:, central_crop:-central_crop, central_crop:-central_crop]), 
    "add", 
    False, 
    pad=32
    )


In [ ]:
pred_amp_object = pred_amp_object / torch.clip(buffer, min=1)
pred_ph_object = pred_ph_object / torch.clip(buffer, min=1)

gt_amp_object = gt_amp_object / torch.clip(buffer, min=1)
gt_ph_object = gt_ph_object / torch.clip(buffer, min=1)

In [ ]:
plt.imshow(buffer, interpolation='none')

In [ ]:
f, ax = plt.subplots(figsize=(9, 8), ncols=2, nrows=2)

gt0 = ax[0, 0].imshow(gt_amp_object, interpolation='none', vmin=0.95, vmax=1, cmap=cmc.oslo)
divider = make_axes_locatable(ax[0, 0])
cax = divider.append_axes('right', size='5%', pad=0.05)
f.colorbar(gt0, cax=cax, orientation='vertical')
ax[0, 0].set_title('GT amplitude')

gt1 = ax[0, 1].imshow(gt_ph_object, interpolation='none', vmin=-0.5, vmax=0.5, cmap=cmc.vik)
divider = make_axes_locatable(ax[0, 1])
cax = divider.append_axes('right', size='5%', pad=0.05)
f.colorbar(gt1, cax=cax, orientation='vertical')
ax[0, 1].set_title('GT phase')

pred0 = ax[1, 0].imshow(pred_amp_object, interpolation='none', vmin=0.95, vmax=1, cmap=cmc.oslo)
divider = make_axes_locatable(ax[1, 0])
cax = divider.append_axes('right', size='5%', pad=0.05)
f.colorbar(pred0, cax=cax, orientation='vertical')
ax[1, 0].set_title('Predicted amplitude')

pred1 = ax[1, 1].imshow(pred_ph_object, interpolation='none', vmin=-1, vmax=0, cmap=cmc.vik)
divider = make_axes_locatable(ax[1, 1])
cax = divider.append_axes('right', size='5%', pad=0.05)
f.colorbar(pred1, cax=cax, orientation='vertical')
ax[1, 1].set_title('Predicted phase')

In [ ]:
pred_amp_object.min(), pred_amp_object.max(), pred_ph_object.min(), pred_ph_object.max()

## Full inference only

In [ ]:
def gen_plot(gt_diff, gt_amp, gt_ph, pred_diff, pred_amp, pred_ph):
    f, ax = plt.subplots(figsize=(7, 4), nrows=2, ncols=3)

    in0 = ax[0, 0].imshow(gt_diff, interpolation='none', norm=colors.LogNorm(vmin=0.1, vmax=np.sqrt(10000)), cmap=cmc.lapaz)
    divider = make_axes_locatable(ax[0, 0])
    cax = divider.append_axes('right', size='5%', pad=0.05)
    f.colorbar(in0, cax=cax, orientation='vertical')
    ax[0, 0].set_title('Input Diff. Amp.')

    in1 = ax[0, 1].imshow(gt_amp, interpolation='none', vmin=0.95, vmax=1, cmap=cmc.oslo)
    divider = make_axes_locatable(ax[0, 1])
    cax = divider.append_axes('right', size='5%', pad=0.05)
    f.colorbar(in1, cax=cax, orientation='vertical', format='%.2f')
    ax[0, 1].set_title('GT Amp.')

    in2 = ax[0, 2].imshow(gt_ph, interpolation='none', vmin=-1, vmax=1, cmap=cmc.vik)
    divider = make_axes_locatable(ax[0, 2])
    cax = divider.append_axes('right', size='5%', pad=0.05)
    f.colorbar(in2, cax=cax, orientation='vertical', format='%.1f')
    ax[0, 2].set_title('GT Phase')

    out0 = ax[1, 0].imshow(pred_diff, interpolation='none', norm=colors.LogNorm(vmin=0.1, vmax=np.sqrt(10000)), 
                           cmap=cmc.lapaz)
    divider = make_axes_locatable(ax[1, 0])
    cax = divider.append_axes('right', size='5%', pad=0.05)
    f.colorbar(out0, cax=cax, orientation='vertical')
    ax[1, 0].set_title('Output Diff. Amp.')

    out1 = ax[1, 1].imshow(pred_amp, interpolation='none', vmin=0.95, vmax=1, cmap=cmc.oslo)
    divider = make_axes_locatable(ax[1, 1])
    cax = divider.append_axes('right', size='5%', pad=0.05)
    f.colorbar(out1, cax=cax, orientation='vertical', format='%.2f')
    ax[1, 1].set_title('Predicted Amp.')

    out2 = ax[1, 2].imshow(pred_ph, interpolation='none', cmap=cmc.vik)
    divider = make_axes_locatable(ax[1, 2])
    cax = divider.append_axes('right', size='5%', pad=0.05)
    f.colorbar(out2, cax=cax, orientation='vertical', format='%.1f')
    ax[1, 2].set_title('Predicted Phase')

    plt.tight_layout()
    return f, ax

In [ ]:
f, ax = gen_plot(gt_diff_test[5000], gt_amp_test[5000], gt_ph_test[5000], 
                 pred_diff_test[5000], pred_amp_test[5000], pred_ph_test[5000])